# ELEC4240: Generative RIS — Colab engineering preflight

The image-generation-derived Marigold backbone directly processes RGB and referring text toward a generated mask target. This notebook checks runtime compatibility only. **It does not train a referring segmentation model or validate a condition-selection policy.**

Select **Runtime → Change runtime type → T4 GPU** (free resources when available), then run the cells. No Drive mount, credentials, private dataset upload, paid service, or background keep-alive is required. A pinned public model is downloaded from Hugging Face. Dtype follows PyTorch support detection, which may include BF16 emulation on T4. The actual dtype is recorded; this does not establish native BF16 acceleration or cross-device numerical equivalence.

The final download preserves the result, environment versions and executed source. Next research gate: real text-mask pairs and a trained baseline that switches target correctly when its expression changes.


In [ ]:
import subprocess, sys, os
os.environ['USE_TF'] = '0'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'diffusers==0.35.2', 'transformers==4.57.1', 'peft==0.17.1', 'accelerate==1.11.0', 'safetensors'], check=True)


In [ ]:
import torch, json, platform, importlib.metadata
from pathlib import Path
assert torch.cuda.is_available(), 'Connect a GPU runtime before running.'
out = Path('/content/generative_ris_v11'); out.mkdir(exist_ok=True)
environment = {'python':platform.python_version(), 'gpu':torch.cuda.get_device_name(), 'cuda':torch.version.cuda, 'packages':{n:importlib.metadata.version(n) for n in ['torch','diffusers','transformers','peft','accelerate']}}
(out/'environment.json').write_text(json.dumps(environment,indent=2))
print(json.dumps(environment,indent=2))


In [ ]:
from pathlib import Path
import subprocess,sys,hashlib
source = '"""Engineering-only probe: no training, no RIS performance claim."""\nimport hashlib\nimport json\nimport time\nfrom pathlib import Path\n\nimport torch\nimport torch.nn.functional as F\n\nfrom diffusers import MarigoldDepthPipeline\nfrom peft import LoraConfig\nimport random\nimport numpy as np\nDTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32\nMODEL_ID = \'prs-eth/marigold-depth-v1-1\'\nREVISION = \'9571e7123e258cf052b4e54241f17971c290e9a8\'\n\ndef setup(seed):\n    assert torch.cuda.is_available(), \'Choose Runtime > Change runtime type > T4 GPU first.\'\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)\n    pipe = MarigoldDepthPipeline.from_pretrained(MODEL_ID, revision=REVISION, variant=\'fp16\', torch_dtype=DTYPE).to(\'cuda\')\n    pipe.vae.requires_grad_(False); pipe.unet.requires_grad_(False); pipe.text_encoder.requires_grad_(False)\n    pipe.unet.add_adapter(LoraConfig(r=4, lora_alpha=4, init_lora_weights=\'gaussian\', target_modules=[\'to_q\',\'to_k\',\'to_v\',\'to_out.0\']))\n    params = {n:p for n,p in pipe.unet.named_parameters() if p.requires_grad}\n    for p in params.values(): p.data = p.data.float()\n    pipe.unet.enable_gradient_checkpointing()\n    pipe.scheduler.set_timesteps(1, device=\'cuda\')\n    assert pipe.scheduler.timesteps.tolist() == [999]\n    assert float(pipe.scheduler.alphas_cumprod[999]) == 0.\n    assert pipe.scheduler.config.prediction_type == \'v_prediction\'\n    return pipe, params\n\ndef encode(pipe, tensor):\n    return pipe.vae.encode(tensor.to(DTYPE)).latent_dist.mode() * pipe.vae.config.scaling_factor\n\n\nROOT = Path(\'/content/generative_ris_v11\')\nOUT = ROOT / \'results\'\n\n\ndef main():\n    OUT.mkdir(parents=True, exist_ok=True)\n    start = time.perf_counter()\n    pipe, params = setup(17)\n    torch.cuda.reset_peak_memory_stats()\n    # Analytic diagnostic inputs, never presented as real referring data.\n    rgb = torch.full((1, 3, 192, 256), -0.7, device=\'cuda\')\n    rgb[:, :, 48:144, 24:104] = torch.tensor([1., -1., -1.], device=\'cuda\')[None, :, None, None]\n    rgb[:, :, 48:144, 152:232] = torch.tensor([-1., -1., 1.], device=\'cuda\')[None, :, None, None]\n    mask = torch.full_like(rgb, -1.)\n    mask[:, :, 48:144, 24:104] = 1.\n    prompts = [\'A binary segmentation mask of the red rectangle, white foreground and black background.\',\n               \'A binary segmentation mask of the blue rectangle, white foreground and black background.\']\n    pipe.unet.eval()\n    with torch.no_grad(), torch.autocast(\'cuda\', dtype=DTYPE, enabled=(DTYPE != torch.float32)):\n        ids = pipe.tokenizer(prompts, padding=\'max_length\', max_length=77,\n                             truncation=True, return_tensors=\'pt\').input_ids.to(\'cuda\')\n        emb = pipe.text_encoder(ids)[0].detach()\n        zrgb, zmask = encode(pipe, rgb).float(), encode(pipe, mask).float()\n        reconstructed = pipe.vae.decode(zmask.to(DTYPE) / pipe.vae.config.scaling_factor).sample.float()\n        noise = torch.randn(zrgb.shape, device=\'cuda\', generator=torch.Generator(device=\'cuda\').manual_seed(91001))\n        inputs = torch.cat([zrgb, noise], 1)\n        red = pipe.unet(inputs, 999, emb[:1]).sample.float()\n        blue = pipe.unet(inputs, 999, emb[1:]).sample.float()\n    pred = reconstructed.mean(1) > 0\n    truth = mask.mean(1) > 0\n    iou = ((pred & truth).sum() / (pred | truth).sum()).item()\n    pipe.unet.train()\n    with torch.autocast(\'cuda\', dtype=DTYPE, enabled=(DTYPE != torch.float32)):\n        velocity = pipe.unet(inputs, 999, emb[:1]).sample.float()\n        loss = F.mse_loss(velocity, -zmask)\n    loss.backward()\n    grads = [p.grad for p in params.values() if p.grad is not None]\n    finite = all(torch.isfinite(g).all().item() for g in grads)\n    norm = sum(g.float().square().sum().item() for g in grads) ** .5\n    torch.cuda.synchronize()\n    result = {\n        \'status\': \'engineering_probe_complete\', \'optimizer_updates\': 0,\n        \'scope\': \'Single analytic RGB/mask pair. Unadapted Marigold with fresh rank-4 LoRA. No object recognition or selector efficacy measured.\',\n        \'gpu\': torch.cuda.get_device_name(), \'dtype\': str(DTYPE), \'model_id\': MODEL_ID, \'model_revision\': REVISION, \'trainable_parameters\': sum(p.numel() for p in params.values()),\n        \'prompts\': prompts, \'embedding_mean_absolute_difference\': (emb[0].float()-emb[1].float()).abs().mean().item(),\n        \'velocity_mean_absolute_difference_same_noise\': (red-blue).abs().mean().item(),\n        \'mask_vae_roundtrip_iou\': iou, \'mask_vae_roundtrip_is_not_prediction\': True,\n        \'latent_supervision_loss\': loss.item(), \'gradient_norm\': norm,\n        \'gradients_finite\': finite, \'parameters_with_grad\': len(grads),\n        \'peak_allocated_mib\': torch.cuda.max_memory_allocated()/2**20,\n        \'seconds_including_load\': time.perf_counter()-start,\n        \'ris_baseline_trained\': False, \'budget_policy_validated\': False,\n        \'source_sha256\': {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest()\n                          for p in [Path(__file__)]}\n    }\n    assert finite and norm > 0 and torch.isfinite(loss).item()\n    (OUT/\'preflight.json\').write_text(json.dumps(result, indent=2)+\'\\n\', encoding=\'utf-8\')\n    print(json.dumps(result, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n'
p = Path('/content/generative_ris_v11/colab_preflight.py')
p.write_text(source)
print('Probe SHA256:',hashlib.sha256(p.read_bytes()).hexdigest())
subprocess.run([sys.executable,str(p)],check=True)


## Interpretation
Finite gradients and prompt-dependent outputs establish a computational path only. A mask VAE round trip is not segmentation prediction accuracy. No optimizer update is performed. Do not interpret this notebook as a trained RIS baseline or as evidence that budgeted verification works.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/generative_ris_v11_preflight', 'zip', '/content/generative_ris_v11')
files.download(archive)
